Este archivo edita los datos crudos y limpios provenientes de 1.Extraccion_ifn y los prepara para subirlos a la bbdd

ENTRADA:
·crudo\IFN2\ifn2_parcela_y_especie.csv
·crudo\IFN2\ifn2_parcela_especie_cd.csv

·crudo\IFN3_campo\ifn3_campo_Brotes.csv
·crudo\IFN3_campo\ifn3_campo_parcela_especie.csv
·crudo\IFN3_campo\ifn3_campo_parcela_mapa.csv
·crudo\IFN3_campo\ifn3_campo_parcela.csv
·crudo\IFN3_sig\ifn3_sig_parcela_especie_cd.csv

·crudo\IFN4_campo\ifn4_campo_Brotes.csv
·crudo\IFN4_campo\ifn4_campo_parcela_especie.csv
·crudo\IFN4_campo\ifn4_campo_parcela_mapa.csv
·crudo\IFN4_campo\ifn4_campo_parcela.csv
·crudo\IFN4_sig\ifn4_sig_parcela_especie_cd.csv

·especies.xlsx (documento creado manualmente que recoge los códigos de las especies de los ifn, sus nombres, sinónimos, equivalencias en los inventarios y clasificación entre frondosa y conífera)

SALIDA
·crudo\parcela.csv
·crudo\parcela_inventario.csv
·crudo\parcela_inventario_especie.csv
·crudo\parcela_inventario_especie_cd.csv

·crudo\ifn4_parcela_inventario.csv
·crudo\ifn4_parcela_especie.csv
·crudo\ifn4_parcela_especie_cd.csv

·crudo\ifn3_parcela_inventario.csv
·crudo\ifn3_parcela_especie.csv
·crudo\ifn3_parcela_especie_cd.csv

·crudo\ifn2_parcela_especie.csv
·crudo\ifn2_parcela_especie_cd.csv

·especies_to_load.csv

-------------
IFN2

Empiezo a crear la tabla para ifn2: saco lo que me interesa

In [1]:
import pandas as pd

# Cargar el archivo CSV
ruta_ifn2 = "IFN_España/IFN2/ifn2_parcela_y_especie.csv"
df_ifn2 = pd.read_csv(ruta_ifn2)

# Crear las nuevas filas para Especie1, Especie2 y Especie3
nuevas_filas = []

# Iterar sobre cada fila y crear tres nuevas filas
for _, row in df_ifn2.iterrows():
    for i in range(1, 4):
        nueva_fila = {
            'idparcela': str(row['IDParcela']),  # Asegurarse de que 'IDParcela' sea una cadena
            'especie': str(row[f'Especie{i}']),  # Asegurarse de que 'Especie' sea una cadena
            'ocupa': row[f'Ocupa{i}'],          # Usar Ocupa1, Ocupa2, Ocupa3
            'estado': row[f'Estado{i}'],        # Usar Estado1, Estado2, Estado3
            'masa': row[f'MASAB{i}'],           # Usar MASAB1, MASAB2, MASAB3
            'origen': row[f'ORIGEB{i}'],        # Usar ORIGEB1, ORIGEB2, ORIGEB3
            'disesp': row['DisEsp'],            # Valor de 'DisEsp' de la fila original
            'comesp': row['ComEsp']            # Valor de 'ComEsp' de la fila original
        }
        nuevas_filas.append(nueva_fila)

# Crear el nuevo DataFrame
nuevo_df = pd.DataFrame(nuevas_filas)

# Crear una clave primaria combinada de IDParcela y Especie
nuevo_df['ClavePrimaria'] = nuevo_df['idparcela'] + '_' + nuevo_df['especie']


# Guardar el nuevo documento en un archivo CSV
nuevo_df.to_csv("IFN_España/ifn2_parcela_especie.csv", index=False)

# Verificar las primeras filas del nuevo DataFrame
print(nuevo_df.head())


  idparcela especie  ocupa  estado  masa  origen  disesp  comesp ClavePrimaria
0   33_1309    72.0    8.0     4.0   3.0     6.0     1.0     2.0  33_1309_72.0
1   33_1309    41.0    1.0     4.0   3.0     6.0     1.0     2.0  33_1309_41.0
2   33_1309    73.0    1.0     4.0   3.0     6.0     1.0     2.0  33_1309_73.0
3   33_1310    28.0    9.0     2.0   2.0     1.0     1.0     2.0  33_1310_28.0
4   33_1310    26.0    1.0     2.0   2.0     1.0     1.0     2.0  33_1310_26.0


Limpio parcela_especie

In [2]:
import pandas as pd
import numpy as np

# Ruta
path = "IFN_España/ifn2_parcela_especie.csv"

# Cargar
df = pd.read_csv(path)

# Función: marca como "malo" NaN, 0 numérico, "0" como texto y cadenas vacías
def is_bad(series: pd.Series) -> pd.Series:
    s = series
    zero_num = pd.to_numeric(s, errors="coerce").eq(0)  # 0 numérico (incluye "0", "0.0")
    zero_str = s.astype(str).str.strip().eq("0")              # "0" textual
    empty   = s.astype(str).str.strip().eq("")                # vacío
    return s.isna() | zero_num | zero_str | empty

bad_id = is_bad(df["idparcela"])
bad_especie = is_bad(df["especie"])
bad_rows = bad_id | bad_especie

print(f"Filas originales: {len(df):,}")
print(f"Filas a eliminar (idparcela/especie NaN o 0): {bad_rows.sum():,}")

# Filtrar y guardar
df_clean = df.loc[~bad_rows].copy()
df_clean.to_csv(path, index=False, encoding="utf-8-sig")

print(f"Filas restantes: {len(df_clean):,}")
print(f"Archivo sobrescrito: {path}")


Filas originales: 23,790
Filas a eliminar (idparcela/especie NaN o 0): 8,714
Filas restantes: 15,076
Archivo sobrescrito: IFN_España/ifn2_parcela_especie.csv


limpio parcela_especie_CD

In [3]:
import pandas as pd
import numpy as np
import os

# Rutas
path_in  = "IFN_España/IFN2/ifn2_IIFL03BD.csv"
path_out = "IFN_España/ifn2_IIFL03BD.csv"

# Cargar
df = pd.read_csv(path_in)

# Normalizar nombres de columnas (quitar espacios y a minúsculas)
df.columns = [c.strip().lower() for c in df.columns]

# Verificar columnas requeridas (idparcela, especie, cd)
requeridas = ["idparcela", "especie", "cd"]
faltantes = [c for c in requeridas if c not in df.columns]
if faltantes:
    raise ValueError(f"Faltan columnas requeridas en el CSV: {faltantes}")

# Función para detectar valores "malos": NaN, 0 numérico, "0" textual, vacío
def is_bad(series: pd.Series) -> pd.Series:
    s = series
    # NaN
    bad_nan = s.isna()
    # 0 numérico (convierte strings a número cuando se pueda)
    num = pd.to_numeric(s, errors="coerce")
    bad_zero_num = num.eq(0)
    # "0" textual y cadenas vacías
    s_str = s.astype(str).str.strip()
    bad_zero_str = s_str.eq("0")
    bad_empty = s_str.eq("")
    return bad_nan | bad_zero_num | bad_zero_str | bad_empty

# Filas a eliminar (si cualquiera de las tres columnas es "mala")
bad_rows = is_bad(df["idparcela"]) | is_bad(df["especie"]) | is_bad(df["cd"])

print(f"Filas originales: {len(df):,}")
print(f"Filas a eliminar (IDParcela/Especie/CD nulos o 0): {bad_rows.sum():,}")

df = df.loc[~bad_rows].copy()

# Eliminar columnas 'Estadillo' y 'Provincia' (ya en minúsculas)
cols_drop = [c for c in ["estadillo", "provincia"] if c in df.columns]
if cols_drop:
    df = df.drop(columns=cols_drop)

# Crear ClavePrimaria con el formato IDParcela_Especie_CD (usando los valores)
# Nota: la columna se llamará 'claveprimaria' porque ya normalizamos a minúsculas
df["claveprimaria"] = (
    df["idparcela"].astype(str).str.strip() + "_" +
    df["especie"].astype(str).str.strip() + "_" +
    df["cd"].astype(str).str.strip()
)

# Asegurar carpeta de salida y guardar
os.makedirs(os.path.dirname(path_out), exist_ok=True)
df.to_csv(path_out, index=False, encoding="utf-8-sig")

print(f"Filas finales: {len(df):,}")
print(f"Guardado en: {path_out}")
print("Columnas finales:", df.columns.tolist())


Filas originales: 289,835
Filas a eliminar (IDParcela/Especie/CD nulos o 0): 0


Filas finales: 289,835
Guardado en: IFN_España/ifn2_IIFL03BD.csv
Columnas finales: ['especie', 'cd', 'npies', 'vcc', 'vsc', 'vle', 'iavc', 'abas', 'idparcela', 'ca', 'cr', 'claveprimaria']


--------
IFN3

ifn3_parcela_especie

In [4]:
import pandas as pd
import numpy as np
import os

# Rutas
path_in  = "IFN_España/IFN3_campo/ifn3_campo_PCEspParc.csv"
path_out = "IFN_España/ifn3_parcela_especie.csv"

# Mapeo de columnas origen -> destino
rename_map = {
    "FPMasa": "fpmasa",
    "Ocupa": "ocupa",
    "PosEsp": "posesp",
    "TratMasa": "tratmasa",
    "Estado": "estado",
    "OrgMasa1": "orgmasa1",
    "IDParcela": "idparcela",
    "EspecieNormalizada": "especie",
}

# Leer CSV origen
df = pd.read_csv(path_in)

# Comprobar columnas disponibles
presentes = [c for c in rename_map.keys() if c in df.columns]
faltantes = [c for c in rename_map.keys() if c not in df.columns]
if faltantes:
    print("[AVISO] No se encontraron en el CSV las columnas:", faltantes)

# Seleccionar y renombrar
df_out = df[presentes].rename(columns=rename_map).copy()

# ---------------- Filtrado solicitado ----------------
filas_originales = len(df_out)

# 1) especie = 0 o NaN (considera "0" como texto también)
especie_num = pd.to_numeric(df_out["especie"], errors="coerce")
mask_bad_especie = especie_num.isna() | especie_num.eq(0)
eliminadas_especie = int(mask_bad_especie.sum())
df_out = df_out.loc[~mask_bad_especie].copy()

# 2) idparcela = NaN (solo NaN, no elimina "0" textual)
mask_bad_id = df_out["idparcela"].isna()
eliminadas_id = int(mask_bad_id.sum())
df_out = df_out.loc[~mask_bad_id].copy()

# Normalizar idparcela (después del filtrado para no convertir NaN a texto)
df_out["idparcela"] = (
    df_out["idparcela"]
    .astype(str).str.strip()
    .str.replace("/.0$", "", regex=True)
)

filas_finales = len(df_out)

print(f"Filas originales: {filas_originales:,}")
print(f"Eliminadas por especie NaN o 0: {eliminadas_especie:,}")
print(f"Eliminadas por idparcela NaN: {eliminadas_id:,}")
print(f"Filas finales: {filas_finales:,}")

# Guardar resultado
os.makedirs(os.path.dirname(path_out), exist_ok=True)
df_out.to_csv(path_out, index=False, encoding="utf-8-sig")

print(f"Archivo generado: {path_out}")
print("Columnas finales:", df_out.columns.tolist())


Filas originales: 147,963
Eliminadas por especie NaN o 0: 0
Eliminadas por idparcela NaN: 0
Filas finales: 147,963


Archivo generado: IFN_España/ifn3_parcela_especie.csv
Columnas finales: ['fpmasa', 'ocupa', 'posesp', 'tratmasa', 'estado', 'orgmasa1', 'idparcela', 'especie']


ifn3_parcela_especie_cd

In [5]:
import pandas as pd
import numpy as np
import os

# Rutas
path_sig   = "IFN_España/IFN3_sig/ifn3_sig_Parcelas_exs.csv"
path_brot  = "IFN_España/IFN3_campo/ifn3_campo_Brotes.csv"
path_out   = "IFN_España/ifn3_parcela_especie_cd.csv"

# ----------------------------
# 1) Leer y seleccionar/renombrar
# ----------------------------
# IFN3 SIG
rename_sig = {
    "CD": "cd",
    "VSC": "vsc",
    "NPies": "npies",
    "ABas": "abas",
    "IAVC": "iavc",
    "VCC": "vcc",
    "VLE": "vle",
    "IDParcela": "idparcela",
    "EspecieNormalizada": "especie",
    "CA": "ca",
    "CR": "cr",
}
use_sig = [c for c in rename_sig if c in pd.read_csv(path_sig, nrows=0).columns]
df_sig = pd.read_csv(path_sig, low_memory=False)
df_sig = df_sig[[c for c in rename_sig if c in df_sig.columns]].rename(columns=rename_sig)

# BROTES
rename_brt = {
    "IDParcela": "idparcela",
    "CD": "cd",
    "NPies": "npies",               # mismo nombre -> luego resolvemos colisión
    "EspecieNormalizada": "especie"
}
use_brt = [c for c in rename_brt if c in pd.read_csv(path_brot, nrows=0).columns]
df_brt = pd.read_csv(path_brot, low_memory=False)
df_brt = df_brt[[c for c in rename_brt if c in df_brt.columns]].rename(columns=rename_brt)

# ----------------------------
# 2) Normalizar claves
# ----------------------------
def clean_id(x: pd.Series) -> pd.Series:
    s = x.astype(str).str.strip()
    s = s.str.replace("/.0$", "", regex=True)  # por si vienen como "123.0"
    s = s.replace({"": np.nan})
    return s

for d in (df_sig, df_brt):
    if "idparcela" in d.columns:
        d["idparcela"] = clean_id(d["idparcela"])
    if "especie" in d.columns:
        d["especie"] = d["especie"].astype(str).str.strip().replace({"": np.nan})
    if "cd" in d.columns:
        d["cd"] = pd.to_numeric(d["cd"], errors="coerce")

# (Opcional) quitar filas sin clave en cada fuente para evitar basura en el merge
df_sig = df_sig.dropna(subset=["idparcela", "especie", "cd"])
df_brt = df_brt.dropna(subset=["idparcela", "especie", "cd"])

# ----------------------------
# 3) Merge por (idparcela, especie, cd)
# ----------------------------
merged = df_sig.merge(
    df_brt,
    on=["idparcela", "especie", "cd"],
    how="outer",
    suffixes=("_sig", "_brotes")
)

# ----------------------------
# 4) Resolver colisión de 'npies'
#    - Preferimos el 'npies' del SIG; si falta, usamos el de Brotes
# ----------------------------
npies_sig = merged.pop("npies_sig") if "npies_sig" in merged.columns else pd.Series(index=merged.index, dtype="float")
npies_brt = merged.pop("npies_brotes") if "npies_brotes" in merged.columns else pd.Series(index=merged.index, dtype="float")

if npies_sig.empty and "npies" in merged.columns:
    # Caso en que no hubo colisión y ya tenemos 'npies' único (p.ej., solo venía de SIG)
    pass
else:
    merged["npies"] = npies_sig
    merged["npies"] = merged["npies"].where(~merged["npies"].isna(), npies_brt)

# ----------------------------
# 5) Orden de columnas (claves primero)
# ----------------------------
key_cols = ["idparcela", "especie", "cd"]
other_cols = [c for c in merged.columns if c not in key_cols]
merged = merged[key_cols + other_cols]

# ----------------------------
# 6) Guardar
# ----------------------------
os.makedirs(os.path.dirname(path_out), exist_ok=True)
merged.to_csv(path_out, index=False, encoding="utf-8-sig")

print(f"Filas unificadas: {len(merged):,} | Columnas: {len(merged.columns):,}")
print(f"Guardado en: {path_out}")
print("Primeras columnas:", merged.columns[:15].tolist())


Filas unificadas: 782,872 | Columnas: 11
Guardado en: IFN_España/ifn3_parcela_especie_cd.csv
Primeras columnas: ['idparcela', 'especie', 'cd', 'vsc', 'abas', 'iavc', 'vcc', 'vle', 'ca', 'cr', 'npies']


------------------
IFN4

ifn4_parcela_inventario

In [6]:
import pandas as pd
import os

# Rutas
path_in  = "IFN_España/IFN4_campo/ifn4_campo_parcela.csv"
path_out = "IFN_España/ifn4_parcela_inventario.csv"

# Variables a extraer (tal como vienen en el CSV de origen)
cols_src = [
    "Ano","TipSuelo1","TipSuelo2","TipSuelo3","Nivel1","Nivel2","Rocosid",
    "MErosiva","Textura","MatOrg","ModComb","DisEsp","ComEsp",
    "FccArb","FccTot","Estado","IDParcela","Radio","Superficie"
]

# Leer
df = pd.read_csv(path_in)

# Comprobar disponibles y faltantes (no rompe si falta alguna)
presentes = [c for c in cols_src if c in df.columns]
faltantes = [c for c in cols_src if c not in df.columns]
if faltantes:
    print("[AVISO] No se encontraron en el CSV las columnas:", faltantes)

# Seleccionar y renombrar a minúsculas
df_out = df[presentes].copy()
df_out.rename(columns={c: c.lower() for c in presentes}, inplace=True)

# ---- Eliminación de filas con idparcela NaN + conteo ----
if "idparcela" not in df_out.columns:
    raise ValueError("No existe la columna 'IDParcela' en el archivo de entrada.")

filas_originales = len(df_out)
na_mask = df_out["idparcela"].isna()
eliminadas_nan = int(na_mask.sum())

df_out = df_out.loc[~na_mask].copy()

# (Opcional) Normalizar idparcela a texto limpio una vez eliminados los NaN
df_out["idparcela"] = (
    df_out["idparcela"]
    .astype(str)
    .str.strip()
    .str.replace("/.0$", "", regex=True)
)

filas_finales = len(df_out)

print(f"Filas originales: {filas_originales:,}")
print(f"Filas eliminadas por idparcela NaN: {eliminadas_nan:,}")
print(f"Filas finales: {filas_finales:,}")

# Guardar
os.makedirs(os.path.dirname(path_out), exist_ok=True)
df_out.to_csv(path_out, index=False, encoding="utf-8-sig")

print(f"Archivo generado: {path_out}")
print("Columnas finales:", df_out.columns.tolist())



[AVISO] No se encontraron en el CSV las columnas: ['Estado']
Filas originales: 49,937
Filas eliminadas por idparcela NaN: 0
Filas finales: 49,937


Archivo generado: IFN_España/ifn4_parcela_inventario.csv
Columnas finales: ['ano', 'tipsuelo1', 'tipsuelo2', 'tipsuelo3', 'nivel1', 'nivel2', 'rocosid', 'merosiva', 'textura', 'matorg', 'modcomb', 'disesp', 'comesp', 'fccarb', 'fcctot', 'idparcela', 'radio', 'superficie']


ifn4_parcela_especie

In [7]:
import pandas as pd
import numpy as np
import os

# Rutas
path_in  = "IFN_España/IFN4_campo/ifn4_campo_PCEspParc.csv"
path_out = "IFN_España/ifn4_parcela_especie.csv"

# Mapeo de columnas origen -> destino
rename_map = {
    "FPMasa": "fpmasa",
    "Ocupa": "ocupa",
    "PosEsp": "posesp",
    "TratMasa": "tratmasa",
    "Estado": "estado",
    "OrgMasa1": "orgmasa1",
    "IDParcela": "idparcela",
    "EspecieNormalizada": "especie",
}

# Leer CSV origen
df = pd.read_csv(path_in)

# Comprobar columnas disponibles
presentes = [c for c in rename_map.keys() if c in df.columns]
faltantes = [c for c in rename_map.keys() if c not in df.columns]
if faltantes:
    print("[AVISO] No se encontraron en el CSV las columnas:", faltantes)

# Seleccionar y renombrar
df_out = df[presentes].rename(columns=rename_map).copy()

# ---------------- Filtrado solicitado ----------------
filas_originales = len(df_out)

# 1) especie = 0 o NaN (considera "0" como texto también)
especie_num = pd.to_numeric(df_out["especie"], errors="coerce")
mask_bad_especie = especie_num.isna() | especie_num.eq(0)
eliminadas_especie = int(mask_bad_especie.sum())
df_out = df_out.loc[~mask_bad_especie].copy()

# 2) idparcela = NaN (solo NaN, no elimina "0" textual)
mask_bad_id = df_out["idparcela"].isna()
eliminadas_id = int(mask_bad_id.sum())
df_out = df_out.loc[~mask_bad_id].copy()

# Normalizar idparcela (después del filtrado para no convertir NaN a texto)
df_out["idparcela"] = (
    df_out["idparcela"]
    .astype(str).str.strip()
    .str.replace("/.0$", "", regex=True)
)

filas_finales = len(df_out)

print(f"Filas originales: {filas_originales:,}")
print(f"Eliminadas por especie NaN o 0: {eliminadas_especie:,}")
print(f"Eliminadas por idparcela NaN: {eliminadas_id:,}")
print(f"Filas finales: {filas_finales:,}")

# Guardar resultado
os.makedirs(os.path.dirname(path_out), exist_ok=True)
df_out.to_csv(path_out, index=False, encoding="utf-8-sig")

print(f"Archivo generado: {path_out}")
print("Columnas finales:", df_out.columns.tolist())


Filas originales: 91,357
Eliminadas por especie NaN o 0: 0
Eliminadas por idparcela NaN: 0
Filas finales: 91,357
Archivo generado: IFN_España/ifn4_parcela_especie.csv
Columnas finales: ['fpmasa', 'ocupa', 'posesp', 'tratmasa', 'estado', 'orgmasa1', 'idparcela', 'especie']


ifn4_parcela_especie_cd

In [8]:
import pandas as pd
import numpy as np
import os

# Rutas
path_sig   = "IFN_España/IFN4_sig/ifn4_sig_Parcelas_exs.csv"
path_brot  = "IFN_España/IFN4_campo/ifn4_campo_Brotes.csv"
path_out   = "IFN_España/ifn4_parcela_especie_cd.csv"

# ----------------------------
# 1) Leer y seleccionar/renombrar
# ----------------------------
# IFN3 SIG
rename_sig = {
    "CD": "cd",
    "VSC": "vsc",
    "NPies": "npies",
    "ABas": "abas",
    "IAVC": "iavc",
    "VCC": "vcc",
    "VLE": "vle",
    "IDParcela": "idparcela",
    "EspecieNormalizada": "especie",
    "CA": "ca",
    "CR": "cr",
}
use_sig = [c for c in rename_sig if c in pd.read_csv(path_sig, nrows=0).columns]
df_sig = pd.read_csv(path_sig, low_memory=False)
df_sig = df_sig[[c for c in rename_sig if c in df_sig.columns]].rename(columns=rename_sig)

# BROTES
rename_brt = {
    "IDParcela": "idparcela",
    "CD": "cd",
    "NPies": "npies",               # mismo nombre -> luego resolvemos colisión
    "EspecieNormalizada": "especie"
}
use_brt = [c for c in rename_brt if c in pd.read_csv(path_brot, nrows=0).columns]
df_brt = pd.read_csv(path_brot, low_memory=False)
df_brt = df_brt[[c for c in rename_brt if c in df_brt.columns]].rename(columns=rename_brt)

# ----------------------------
# 2) Normalizar claves
# ----------------------------
def clean_id(x: pd.Series) -> pd.Series:
    s = x.astype(str).str.strip()
    s = s.str.replace("/.0$", "", regex=True)  # por si vienen como "123.0"
    s = s.replace({"": np.nan})
    return s

for d in (df_sig, df_brt):
    if "idparcela" in d.columns:
        d["idparcela"] = clean_id(d["idparcela"])
    if "especie" in d.columns:
        d["especie"] = d["especie"].astype(str).str.strip().replace({"": np.nan})
    if "cd" in d.columns:
        d["cd"] = pd.to_numeric(d["cd"], errors="coerce")

# (Opcional) quitar filas sin clave en cada fuente para evitar basura en el merge
df_sig = df_sig.dropna(subset=["idparcela", "especie", "cd"])
df_brt = df_brt.dropna(subset=["idparcela", "especie", "cd"])

# ----------------------------
# 3) Merge por (idparcela, especie, cd)
# ----------------------------
merged = df_sig.merge(
    df_brt,
    on=["idparcela", "especie", "cd"],
    how="outer",
    suffixes=("_sig", "_brotes")
)

# ----------------------------
# 4) Resolver colisión de 'npies'
#    - Preferimos el 'npies' del SIG; si falta, usamos el de Brotes
# ----------------------------
npies_sig = merged.pop("npies_sig") if "npies_sig" in merged.columns else pd.Series(index=merged.index, dtype="float")
npies_brt = merged.pop("npies_brotes") if "npies_brotes" in merged.columns else pd.Series(index=merged.index, dtype="float")

if npies_sig.empty and "npies" in merged.columns:
    # Caso en que no hubo colisión y ya tenemos 'npies' único (p.ej., solo venía de SIG)
    pass
else:
    merged["npies"] = npies_sig
    merged["npies"] = merged["npies"].where(~merged["npies"].isna(), npies_brt)

# ----------------------------
# 5) Orden de columnas (claves primero)
# ----------------------------
key_cols = ["idparcela", "especie", "cd"]
other_cols = [c for c in merged.columns if c not in key_cols]
merged = merged[key_cols + other_cols]

# ----------------------------
# 6) Guardar
# ----------------------------
os.makedirs(os.path.dirname(path_out), exist_ok=True)
merged.to_csv(path_out, index=False, encoding="utf-8-sig")

print(f"Filas unificadas: {len(merged):,} | Columnas: {len(merged.columns):,}")
print(f"Guardado en: {path_out}")
print("Primeras columnas:", merged.columns[:15].tolist())


Filas unificadas: 533,466 | Columnas: 11
Guardado en: IFN_España/ifn4_parcela_especie_cd.csv
Primeras columnas: ['idparcela', 'especie', 'cd', 'vsc', 'abas', 'iavc', 'vcc', 'vle', 'ca', 'cr', 'npies']


--------
Unificar

parcela_inventario_especie_cd

In [9]:
import pandas as pd
import numpy as np
import os

# Rutas
p_ifn2 = "IFN_España/ifn2_IIFL03BD.csv"
p_ifn3 = "IFN_España/ifn3_parcela_especie_cd.csv"
p_ifn4 = "IFN_España/ifn4_parcela_especie_cd.csv"
p_out  = "IFN_España/parcela_inventario_especie_cd.csv"

# Columnas del resultado final (en este orden)
FINAL_COLS = ["idparcela", "inventario", "especie", "cd",
              "npies", "vcc", "vsc", "vle", "iavc", "abas", "ca", "cr"]

# Columnas a rellenar con 0
FILL_ZERO_COLS = ["vsc", "abas", "iavc", "vcc", "vle", "ca", "cr"]

def clean_idparcela(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    return s.str.replace("/.0$", "", regex=True).replace({"": np.nan})

def load_prepare(path: str, inventario_value: int) -> pd.DataFrame:
    df = pd.read_csv(path)
    # normalizar nombres de columnas a minúsculas
    df.columns = [c.strip().lower() for c in df.columns]

    # verificar claves mínimas
    required_min = {"idparcela", "especie", "cd"}
    missing = required_min - set(df.columns)
    if missing:
        raise ValueError(f"[{path}] Faltan columnas requeridas: {sorted(missing)}")

    # asegurar columnas del resultado (crear si faltan)
    for c in FINAL_COLS:
        if c not in df.columns:
            df[c] = pd.NA

    # limpiar tipos clave
    df["idparcela"] = clean_idparcela(df["idparcela"])
    df["especie"]   = df["especie"].astype(str).str.strip().replace({"": np.nan})
    df["cd"]        = pd.to_numeric(df["cd"], errors="coerce")

    # rellenar con 0 los solicitados (si hay NaN)
    for c in FILL_ZERO_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    # inventario
    df["inventario"] = inventario_value

    # quedarnos solo con las columnas finales en el orden deseado
    return df[FINAL_COLS].copy()

# Cargar y preparar cada fuente
df2 = load_prepare(p_ifn2, inventario_value=2)
df3 = load_prepare(p_ifn3, inventario_value=3)
df4 = load_prepare(p_ifn4, inventario_value=4)

# Unificar
df_all = pd.concat([df2, df3, df4], ignore_index=True)

# (Opcional) eliminar filas con idparcela/especie/cd faltantes
# df_all = df_all.dropna(subset=["idparcela", "especie", "cd"])

# Guardar
os.makedirs(os.path.dirname(p_out), exist_ok=True)
df_all.to_csv(p_out, index=False, encoding="utf-8-sig")

print(f"IFN2: {len(df2):,} | IFN3: {len(df3):,} | IFN4: {len(df4):,}")
print(f"Total unificado: {len(df_all):,} filas  |  {len(df_all.columns)} columnas")
print(f"Guardado en: {p_out}")


/tmp/ipykernel_2046064/365012874.py:23: DtypeWarning: Columns (0: especie) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


IFN2: 289,835 | IFN3: 782,872 | IFN4: 533,466
Total unificado: 1,606,173 filas  |  12 columnas
Guardado en: IFN_España/parcela_inventario_especie_cd.csv


In [10]:
# -*- coding: utf-8 -*-
import pandas as pd
from pathlib import Path

# -------- Ruta --------
file_cd = "IFN_España/parcela_inventario_especie_cd.csv"
p = Path(file_cd)
outfile = "IFN_España/parcela_inventario_especie_cd.csv"

# -------- Leer --------
df = pd.read_csv(file_cd, low_memory=False)
print(f"Filas originales: {len(df):,}")

# -------- Clave natural --------
key_cols = ["idparcela", "inventario", "especie", "cd"]

# Ensure key_cols exist
missing_keys = [c for c in key_cols if c not in df.columns]
if missing_keys:
    print(f"[AVISO] Faltan columnas clave: {missing_keys}")
    key_cols = [c for c in key_cols if c in df.columns]

# 1) Eliminar duplicados exactos (todas las columnas idénticas)
df_noexact = df.drop_duplicates()
print(f"Tras eliminar duplicados exactos: {len(df_noexact):,} filas")

# 2) Resolver duplicados por clave (npies = suma)
# Pandas 2.0-compatible: use agg instead of apply
other_cols = [c for c in df_noexact.columns if c not in key_cols and c != "npies"]
agg_dict = {"npies": "sum"}
for c in other_cols:
    agg_dict[c] = "first"

df_unified = (
    df_noexact.groupby(key_cols, dropna=False, sort=False)
              .agg(agg_dict)
              .reset_index()
)

print(f"Tras unificar npies en duplicados por clave: {len(df_unified):,} filas")

# 3) Diagnóstico de duplicados restantes
grp_size = df_unified.groupby(key_cols, dropna=False).size()
dup_keys = grp_size[grp_size > 1]
print("\n===== Diagnóstico final =====")
print(f"Claves duplicadas restantes: {len(dup_keys):,}")
print(f"Filas involucradas en duplicados restantes: {dup_keys.sum():,}")

# 4) Guardar
df_unified.to_csv(outfile, index=False, encoding="utf-8-sig")
print(f"\n✅ Archivo limpio guardado en:\n{outfile}")


Filas originales: 1,606,173


Tras eliminar duplicados exactos: 1,528,475 filas


Tras unificar npies en duplicados por clave: 1,475,707 filas



===== Diagnóstico final =====
Claves duplicadas restantes: 0
Filas involucradas en duplicados restantes: 0



✅ Archivo limpio guardado en:
IFN_España/parcela_inventario_especie_cd.csv


cambios en la especie para que no de problemas

In [11]:
# -*- coding: utf-8 -*-
import pandas as pd

# archivo de entrada
archivo_csv = "IFN_España/parcela_inventario_especie_cd.csv"
archivo_salida = "IFN_España/parcela_inventario_especie_cd.csv"

# leer CSV
df = pd.read_csv(archivo_csv, dtype=str)

# diccionario de reemplazos
map_especie = {
    "350": "50",
    "526": "26",
    "626": "26",
    "726": "26",
    "826": "26",
    "926": "26",
    "646": "46",
    "746": "46",
    "846": "46",
    "946": "46",
    "837": "37",
    "919": "19",
    "958": "58",
    "989": "89",
    "990": "90",
}

# asegurar que la columna existe
if "especie" not in df.columns:
    raise ValueError("El archivo no tiene columna 'especie'")

# aplicar reemplazos
df["especie"] = df["especie"].astype(str).replace(map_especie)

# guardar resultado
df.to_csv(archivo_salida, index=False, encoding="utf-8")
print(f"✅ Edición terminada. Archivo guardado en: {archivo_salida}")


✅ Edición terminada. Archivo guardado en: IFN_España/parcela_inventario_especie_cd.csv


In [12]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

# -------- CONFIG --------
IN_PATH  = Path("IFN_España/parcela_inventario_especie_cd.csv")
OUT_PATH = Path("IFN_España/parcela_inventario_especie_cd.csv")  # salida limpia
BACKUP_PATH = IN_PATH.with_suffix(".csv.bak")                      # copia de seguridad opcional

# Claves de identificación (deben quedar únicas)
KEYS = ["idparcela", "inventario", "especie", "cd"]

# Columnas a sumar si hay duplicados
NUM_COLS = ["npies", "vcc", "vsc", "vle", "iavc", "abas", "ca", "cr"]

# -------- LECTURA --------
print(f"📥 Leyendo: {IN_PATH}")
df = pd.read_csv(IN_PATH, dtype=str)  # leemos todo como string para normalizar nosotros
print(f"  Filas leídas: {len(df):,}")

# -------- NORMALIZACIÓN TIPOS --------
# Asegurar que las claves existan
missing_cols = [c for c in KEYS + NUM_COLS if c not in df.columns]
if missing_cols:
    raise SystemExit(f"Faltan columnas en el CSV: {missing_cols}")

# Limpiar espacios en claves y mantenerlas como strings
for c in KEYS:
    df[c] = df[c].astype(str).str.strip()

# Coerción numérica segura en las columnas a sumar
def to_num(series):
    # reemplazar comas decimales -> puntos, quitar espacios
    s = series.astype(str).str.replace(",", ".", regex=False).str.strip()
    return pd.to_numeric(s, errors="coerce")

for c in NUM_COLS:
    df[c] = to_num(df[c])

# Rellenar NaN con 0 para poder sumar
df[NUM_COLS] = df[NUM_COLS].fillna(0)

# -------- DEDUP (AGRUPAR + SUMAR) --------
before_rows = len(df)
dups_count = df.duplicated(subset=KEYS, keep=False).sum()
print(f"🔎 Filas pertenecientes a combinaciones duplicadas (antes): {dups_count:,}")

# Agrupar: suma de numéricas, y nos quedamos con las claves tal cual
agg_dict = {c: "sum" for c in NUM_COLS}
df_sum = (
    df.groupby(KEYS, as_index=False)
      .agg(agg_dict)
)

after_rows = len(df_sum)
removed = before_rows - after_rows
print(f"✅ Agrupado por {KEYS}.")
print(f"  Filas originales : {before_rows:,}")
print(f"  Filas tras dedup : {after_rows:,}")
print(f"  Filas “colapsadas” (por duplicados): {removed:,}")

# -------- VALIDACIONES --------
# 1) Comprobar unicidad de las claves en la salida
still_dups = df_sum.duplicated(subset=KEYS).sum()
if still_dups > 0:
    print(f"⚠️ Aún hay duplicados según {KEYS}: {still_dups}")
else:
    print("🧩 Unicidad verificada: no quedan duplicados por las claves.")

# 2) Chequeo rápido de rangos numéricos tras sumar
desc = df_sum[NUM_COLS].describe().T
print("\n📊 Resumen de columnas sumadas después de la consolidación:")
print(desc.to_string())


# Guardar el archivo deduplicado (no sobrescribimos el original)
df_sum.to_csv(OUT_PATH, index=False, encoding="utf-8")
print(f"📤 Archivo deduplicado escrito en: {OUT_PATH}")


📥 Leyendo: IFN_España/parcela_inventario_especie_cd.csv


  Filas leídas: 1,475,707


🔎 Filas pertenecientes a combinaciones duplicadas (antes): 2,957


✅ Agrupado por ['idparcela', 'inventario', 'especie', 'cd'].
  Filas originales : 1,475,707
  Filas tras dedup : 1,474,207
  Filas “colapsadas” (por duplicados): 1,500
🧩 Unicidad verificada: no quedan duplicados por las claves.



📊 Resumen de columnas sumadas después de la consolidación:
           count       mean         std        min       25%        50%        75%          max
npies  1474207.0  64.496494  152.656391   0.000000  5.092962  15.000000  56.588472  6238.879039
vcc    1474207.0   9.467731   17.076517   0.000000  0.000000   3.549989  10.920594   676.849063
vsc    1474207.0   7.335912   13.672137  -5.982382  0.000000   2.646015   8.328947   590.539861
vle    1474207.0   0.797541    1.853740   0.000000  0.000000   0.325950   0.972966   295.598884
iavc   1474207.0   0.459476    1.195643 -69.048779  0.000000   0.112574   0.415952    68.949877
abas   1474207.0   1.748813    2.691567   0.000000  0.000000   0.940900   2.164450   115.958556
ca     1474207.0   4.036051    7.086291   0.000000  0.000000   1.920652   5.062303   560.930285
cr     1474207.0   1.690217    5.655389   0.000000  0.000000   0.702770   1.842236   584.387777


📤 Archivo deduplicado escrito en: IFN_España/parcela_inventario_especie_cd.csv


parcela_inventario_especie

In [13]:
import pandas as pd
import numpy as np
import os

# Rutas de entrada y salida
p_ifn2 = "IFN_España/ifn2_parcela_especie.csv"
p_ifn3 = "IFN_España/ifn3_parcela_especie.csv"
p_ifn4 = "IFN_España/ifn4_parcela_especie.csv"
p_out  = "IFN_España/parcela_inventario_especie.csv"

# Columnas "comunes" de la salida
COMMON_COLS = [
    "idparcela", "especie", "ocupa", "estado", "fpmasa", "posesp",
    "tratmasa", "orgmasa1", "masa", "origen", "disesp", "comesp"
]

def load_align(path: str, inventario: int) -> pd.DataFrame:
    df = pd.read_csv(path)
    # normaliza nombres a minúsculas sin espacios
    df.columns = [c.strip().lower() for c in df.columns]

    # comprueba claves mínimas
    missing_keys = {"idparcela", "especie"} - set(df.columns)
    if missing_keys:
        raise ValueError(f"[{path}] Faltan columnas clave: {sorted(missing_keys)}")

    # crea las columnas que falten y respeta las existentes
    for c in COMMON_COLS:
        if c not in df.columns:
            df[c] = pd.NA

    # selecciona solo las columnas de interés en el orden dado
    df = df[COMMON_COLS].copy()

    # normaliza idparcela/especie para consistencia
    df["idparcela"] = (
        df["idparcela"].astype(str).str.strip()
        .str.replace("/.0$", "", regex=True)
        .replace({"": np.nan})
    )
    df["especie"] = df["especie"].astype(str).str.strip().replace({"": np.nan})

    # añade el inventario
    df["inventario"] = inventario

    # reordena: idparcela, inventario, especie, resto
    ordered = ["idparcela", "inventario", "especie"] + [c for c in COMMON_COLS if c not in ["idparcela", "especie"]]
    return df[ordered]

# Carga y alinea cada fuente
df2 = load_align(p_ifn2, inventario=2)
df3 = load_align(p_ifn3, inventario=3)
df4 = load_align(p_ifn4, inventario=4)

# Unifica (apila filas)
df_all = pd.concat([df2, df3, df4], ignore_index=True)

# (Opcional) eliminar filas sin clave tras normalización
# df_all = df_all.dropna(subset=["idparcela", "especie"])

# Guarda
os.makedirs(os.path.dirname(p_out), exist_ok=True)
df_all.to_csv(p_out, index=False, encoding="utf-8-sig")

print(f"IFN2: {len(df2):,} | IFN3: {len(df3):,} | IFN4: {len(df4):,}")
print(f"Total unificado: {len(df_all):,} filas")
print(f"Guardado en: {p_out}")
print("Columnas finales:", df_all.columns.tolist())


IFN2: 15,076 | IFN3: 147,963 | IFN4: 91,357
Total unificado: 254,396 filas
Guardado en: IFN_España/parcela_inventario_especie.csv
Columnas finales: ['idparcela', 'inventario', 'especie', 'ocupa', 'estado', 'fpmasa', 'posesp', 'tratmasa', 'orgmasa1', 'masa', 'origen', 'disesp', 'comesp']


In [14]:
import pandas as pd

# Rutas de los archivos
file_cd = "IFN_España/parcela_inventario_especie_cd.csv"
file_especie = "IFN_España/parcela_inventario_especie.csv"

# Leer archivos
df_cd = pd.read_csv(file_cd, sep=",")
df_especie = pd.read_csv(file_especie, sep=",")

# Agrupar sumando ca y cr en cd
df_cd_sum = (
    df_cd.groupby(["idparcela", "inventario", "especie"], as_index=False)[["ca", "cr"]]
    .sum()
)

# Hacer merge con especie
df_final = df_especie.drop(columns=["ca", "cr", "c"], errors="ignore").merge(
    df_cd_sum, on=["idparcela", "inventario", "especie"], how="left"
)

# Calcular c = ca + cr
df_final["c"] = df_final["ca"] + df_final["cr"]

# Guardar el resultado
output_file = "IFN_España/parcela_inventario_especie.csv"
df_final.to_csv(output_file, index=False)

print(f"Archivo procesado guardado en: {output_file}")


/tmp/ipykernel_2046064/1869288379.py:8: DtypeWarning: Columns (0: especie) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cd = pd.read_csv(file_cd, sep=",")


Archivo procesado guardado en: IFN_España/parcela_inventario_especie.csv


In [15]:
# -*- coding: utf-8 -*-
import pandas as pd
from pathlib import Path

# -------- Parámetros --------
infile = "IFN_España/parcela_inventario_especie.csv"
p = Path(infile)
outfile = "IFN_España/parcela_inventario_especie.csv"

# -------- Cargar --------
df = pd.read_csv(infile)
print(f"Filas originales: {len(df):,}")

# -------- Comprobación columnas necesarias --------
key_cols = ["idparcela", "inventario", "especie"]
sum_cols = ["ocupa", "ca", "cr", "c"]
winner_cols = ["fpmasa", "posesp", "tratmasa", "orgmasa1", "masa", "origen", "disesp", "comesp", "estado"]  # estado se sobrescribe luego por regla

needed = set(key_cols + sum_cols + winner_cols)
missing = needed - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas en el CSV: {sorted(missing)}")

# -------- Detectar duplicados por clave --------
grp_size = df.groupby(key_cols, dropna=False).size().rename("n")
dup_keys = grp_size[grp_size > 1]
n_dup_keys = len(dup_keys)
rows_in_dup_keys = dup_keys.sum()
print(f"Claves duplicadas (idparcela,inventario,especie) con >1 fila: {n_dup_keys:,} (filas afectadas: {rows_in_dup_keys:,})")

# -------- Agregados por grupo (sumas) --------
agg_sum = (
    df.groupby(key_cols, dropna=False)[sum_cols]
      .sum(min_count=1)  # si todo es NaN, queda NaN
      .reset_index()
      .rename(columns={
          "ocupa": "ocupa_unificada",
          "ca": "ca_unificada",
          "cr": "cr_unificada",
          "c": "c_unificada",
      })
)

# -------- Selección de fila ganadora por grupo --------
# Orden: mayor ocupa, y si empata, menor estado
df_sorted = df.sort_values(key_cols + ["ocupa", "estado"], ascending=[True, True, True, False, True])
winners = (
    df_sorted.drop_duplicates(subset=key_cols, keep="first")
             [key_cols + winner_cols]  # incluye estado de la fila ganadora
)

# -------- Construir unificado --------
unificado = winners.merge(agg_sum, on=key_cols, how="inner")

# Reglas finales:
# - estado = estado de la fila ganadora (ya viene en winners)
# - ocupa/ca/cr/c = agregados sumados
unificado = (
    unificado
    .assign(
        ocupa=lambda d: d["ocupa_unificada"],
        ca=lambda d: d["ca_unificada"],
        cr=lambda d: d["cr_unificada"],
        c=lambda d: d["c_unificada"],
    )
    .drop(columns=["ocupa_unificada","ca_unificada","cr_unificada","c_unificada"])
)

# Mantener también cualquier otra columna que existiera y no esté en las reglas:
other_cols = [c for c in df.columns if c not in set(unificado.columns) | set(key_cols)]
# (aquí podrían combinarse más columnas con la fila ganadora)

# -------- Conteos antes/después --------
rows_before = len(df)
rows_after = len(unificado)
rows_removed = rows_before - rows_after

print("\n===== RESUMEN UNIFICACIÓN =====")
print(f"Filas antes : {rows_before:,}")
print(f"Filas después: {rows_after:,}")
print(f"Filas eliminadas por unificación: {rows_removed:,}")

# Para verificar: número de claves únicas antes/después
unique_keys_before = df[key_cols].drop_duplicates().shape[0]
unique_keys_after  = unificado[key_cols].drop_duplicates().shape[0]
print(f"Claves únicas antes : {unique_keys_before:,}")
print(f"Claves únicas después: {unique_keys_after:,}")

# -------- Guardar --------
unificado.to_csv(outfile, index=False, encoding="utf-8-sig")
print(f"\n✅ Archivo unificado guardado en:\n{outfile}")


Filas originales: 254,396
Claves duplicadas (idparcela,inventario,especie) con >1 fila: 147 (filas afectadas: 298)



===== RESUMEN UNIFICACIÓN =====
Filas antes : 254,396
Filas después: 254,245
Filas eliminadas por unificación: 151
Claves únicas antes : 254,245
Claves únicas después: 254,245



✅ Archivo unificado guardado en:
IFN_España/parcela_inventario_especie.csv


parcela_inventario.csv

In [16]:
import pandas as pd
import numpy as np
import os

# Rutas de entrada
p_ifn2 = "IFN_España/IFN2/ifn2_parcela_y_especie.csv"
p_ifn3 = "IFN_España/IFN3_campo/ifn3_campo_parcela.csv"
p_ifn4 = "IFN_España/IFN4_campo/ifn4_campo_parcela.csv"

# Salida
p_out  = "IFN_España/parcela_inventario.csv"

# Columnas a extraer (tal cual en los CSV de origen IFN3/IFN4)
COLS_SRC = [
    "IDParcela","Ano","Nivel1","Nivel2","MErosiva","Textura","MatOrg","ModComb",
    "DisEsp","ComEsp","FccArb","FccTot","Estado"
]

# Para IFN2 solo existen estas (más IDParcela):
COLS_IFN2 = ["IDParcela", "Ano", "MErosiva", "DisEsp", "ComEsp"]

def load_subset(path: str, inventario_value: int, cols_src: list) -> pd.DataFrame:
    df = pd.read_csv(path)

    # Aviso si faltan columnas (no rompe)
    present = [c for c in cols_src if c in df.columns]
    missing = [c for c in cols_src if c not in df.columns]
    if missing:
        print(f"[AVISO] En {os.path.basename(path)} faltan columnas: {missing}")

    # Selección y renombrado a minúsculas
    df = df[present].copy()
    df.rename(columns={c: c.lower() for c in present}, inplace=True)

    # Normaliza idparcela si existe (quita espacios / '.0')
    if "idparcela" in df.columns:
        df["idparcela"] = (
            df["idparcela"]
            .astype(str).str.strip()
            .str.replace("/.0$", "", regex=True)
            .replace({"": np.nan})
        )

    # Añade inventario
    df["inventario"] = inventario_value

    # Deduplicar por idparcela (IFN2 suele venir repetido por especie)
    if "idparcela" in df.columns:
        df = df.drop_duplicates(subset=["idparcela"], keep="first")

    # Reordenar: columnas pedidas en minúsculas + inventario (de las que estén presentes)
    ordered = [c.lower() for c in cols_src] + ["inventario"]
    ordered_present = [c for c in ordered if c in df.columns]
    return df[ordered_present]

# Carga y prepara cada fuente
df2 = load_subset(p_ifn2, inventario_value=2, cols_src=COLS_IFN2)
df3 = load_subset(p_ifn3, inventario_value=3, cols_src=COLS_SRC)
df4 = load_subset(p_ifn4, inventario_value=4, cols_src=COLS_SRC)

# Unifica (apila filas)
df_all = pd.concat([df2, df3, df4], ignore_index=True)

# Orden final sugerido (unión de todas las columnas habituales)
final_order = [
    "idparcela","inventario","ano","nivel1","nivel2","merosiva","textura","matorg",
    "modcomb","disesp","comesp","fccarb","fcctot","estado"
]
# Mantén las que existan y añade cualquier otra que haya aparecido
ordered_present = [c for c in final_order if c in df_all.columns]
other_cols = [c for c in df_all.columns if c not in ordered_present]
df_all = df_all[ordered_present + other_cols]

# Guardar
os.makedirs(os.path.dirname(p_out), exist_ok=True)
df_all.to_csv(p_out, index=False, encoding="utf-8-sig")

print(f"IFN2 filas: {len(df2):,} | IFN3 filas: {len(df3):,} | IFN4 filas: {len(df4):,}")
print(f"Total unificado: {len(df_all):,} filas | {len(df_all.columns)} columnas")
print(f"Guardado en: {p_out}")
print("Columnas finales:", df_all.columns.tolist())


[AVISO] En ifn4_campo_parcela.csv faltan columnas: ['Estado']


/tmp/ipykernel_2046064/3006672013.py:23: DtypeWarning: Columns (0: Nivel2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


IFN2 filas: 7,930 | IFN3 filas: 82,022 | IFN4 filas: 49,937
Total unificado: 139,889 filas | 14 columnas
Guardado en: IFN_España/parcela_inventario.csv
Columnas finales: ['idparcela', 'inventario', 'ano', 'nivel1', 'nivel2', 'merosiva', 'textura', 'matorg', 'modcomb', 'disesp', 'comesp', 'fccarb', 'fcctot', 'estado']


parcela.csv

In [17]:
import pandas as pd
import numpy as np
import os

# -----------------------------
# Rutas de entrada/salida
# -----------------------------
p_ifn2_parc_esp   = "IFN_España/IFN2/ifn2_parcela_y_especie.csv"
p_ifn3_mapa       = "IFN_España/IFN3_campo/ifn3_campo_PCEspMapa.csv"
p_ifn4_mapa       = "IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv"
p_ifn4_parcela    = "IFN_España/IFN4_campo/ifn4_campo_parcela.csv"
p_out             = "IFN_España/parcela.csv"

# -----------------------------
# Utilidades
# -----------------------------
def clean_idparcela(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    s = s.str.replace("/.0$", "", regex=True)
    s = s.replace({"": np.nan})
    return s

# -----------------------------
# 1) Presencia por inventario
# -----------------------------
# IFN2 (puede traer múltiples por especie; nos quedamos con IDParcela único)
df2 = pd.read_csv(p_ifn2_parc_esp, usecols=lambda c: c == "IDParcela" or c.lower() == "idparcela")
df2.columns = [c.strip().lower() for c in df2.columns]
if "idparcela" not in df2.columns:
    raise ValueError(f"[{p_ifn2_parc_esp}] No encuentro la columna 'IDParcela'.")
df2["idparcela"] = clean_idparcela(df2["idparcela"])
set2 = set(df2.dropna(subset=["idparcela"])["idparcela"].unique())

# IFN3 mapa
df3m = pd.read_csv(p_ifn3_mapa, usecols=lambda c: c == "IDParcela" or c.lower() == "idparcela")
df3m.columns = [c.strip().lower() for c in df3m.columns]
if "idparcela" not in df3m.columns:
    raise ValueError(f"[{p_ifn3_mapa}] No encuentro la columna 'IDParcela'.")
df3m["idparcela"] = clean_idparcela(df3m["idparcela"])
set3 = set(df3m.dropna(subset=["idparcela"])["idparcela"].unique())

# IFN4 mapa (también lo usaré para las coordenadas)
df4m = pd.read_csv(p_ifn4_mapa)
df4m.columns = [c.strip().lower() for c in df4m.columns]
if "idparcela" not in df4m.columns:
    raise ValueError(f"[{p_ifn4_mapa}] No encuentro la columna 'IDParcela'.")
df4m["idparcela"] = clean_idparcela(df4m["idparcela"])
set4 = set(df4m.dropna(subset=["idparcela"])["idparcela"].unique())

# -----------------------------
# 2) Variable presencia (1/2/3)
# -----------------------------
# Base sobre IFN4 (porque toda la salida necesita datos de IFN4)
pres_df = pd.DataFrame({"idparcela": sorted(set4)})
pres_df["in2"] = pres_df["idparcela"].isin(set2)
pres_df["in3"] = pres_df["idparcela"].isin(set3)
pres_df["in4"] = pres_df["idparcela"].isin(set4)  # siempre True aquí, pero por claridad

# Reglas:
# 1: en 2 & 3 & 4
# 2: en 2 & 4 (independientemente de 3) -> si también en 3, la regla 1 prevalece
# 3: en 3 & 4 (independientemente de 2) -> si también en 2, la regla 1 prevalece
cond1 = pres_df["in2"] & pres_df["in3"] & pres_df["in4"]
cond2 = pres_df["in2"] & pres_df["in4"]
cond3 = pres_df["in3"] & pres_df["in4"]

pres_df["presencia"] = np.select(
    [cond1, cond2, cond3],
    [1, 2, 3],
    default=np.nan
)

# Filtrar solo las que cumplen (IFN4 y (IFN2 o IFN3))
pres_df = pres_df.dropna(subset=["presencia"]).copy()
pres_df["presencia"] = pres_df["presencia"].astype("Int64")

# -----------------------------
# 3) Datos desde IFN4
# -----------------------------
# 3a) IFN4 mapa -> coorx, coory, huso
mapa_cols = {"idparcela", "coorx", "coory", "huso"}
rename_mapa = {"CoorX": "coorx", "CoorY": "coory", "Huso": "huso", "IDParcela": "idparcela"}
# si no están ya en minúsculas:
df4m = pd.read_csv(p_ifn4_mapa)
df4m = df4m.rename(columns=rename_mapa)
df4m.columns = [c.strip().lower() for c in df4m.columns]
keep_mapa = [c for c in mapa_cols if c in df4m.columns]
df4m = df4m[keep_mapa].copy()
df4m["idparcela"] = clean_idparcela(df4m["idparcela"])
df4m = df4m.drop_duplicates(subset=["idparcela"], keep="first")

# 3b) IFN4 parcela -> tipsuelo1, tipsuelo2, tipsuelo3, rocosidad, radio, superficie
rename_parc4 = {
    "IDParcela": "idparcela",
    "TipSuelo1": "tipsuelo1",
    "TipSuelo2": "tipsuelo2",
    "TipSuelo3": "tipsuelo3",
    "Rocosid": "rocosidad",
    "Radio": "radio",
    "Superficie": "superficie",
}
df4p = pd.read_csv(p_ifn4_parcela)
df4p = df4p.rename(columns=rename_parc4)
df4p.columns = [c.strip().lower() for c in df4p.columns]
keep_parc4 = [c for c in rename_parc4.values() if c in df4p.columns]
df4p = df4p[keep_parc4].copy()
df4p["idparcela"] = clean_idparcela(df4p["idparcela"])
df4p = df4p.drop_duplicates(subset=["idparcela"], keep="first")

# -----------------------------
# 4) Unir y guardar
# -----------------------------
# Unir datos IFN4 (mapa + parcela)
df_ifn4 = df4m.merge(df4p, on="idparcela", how="left")

# Añadir presencia
out = pres_df[["idparcela", "presencia"]].merge(df_ifn4, on="idparcela", how="left")

# Reordenar columnas finales
final_cols = ["idparcela", "coorx", "coory", "huso", "presencia",
              "tipsuelo1", "tipsuelo2", "tipsuelo3", "rocosidad", "radio", "superficie"]
final_present = [c for c in final_cols if c in out.columns]
out = out[final_present].copy()

# Guardar
os.makedirs(os.path.dirname(p_out), exist_ok=True)
out.to_csv(p_out, index=False, encoding="utf-8-sig")

print(f"Parcelas IFN2: {len(set2):,} | IFN3: {len(set3):,} | IFN4: {len(set4):,}")
print("Distribución 'presencia':")
print(out["presencia"].value_counts(dropna=False).sort_index())
print(f"Filas finales: {len(out):,}  |  Guardado en: {p_out}")
print("Columnas finales:", out.columns.tolist())


Parcelas IFN2: 7,930 | IFN3: 91,782 | IFN4: 50,643
Distribución 'presencia':
presencia
1     7094
2      181
3    33726
Name: count, dtype: Int64
Filas finales: 41,001  |  Guardado en: IFN_España/parcela.csv
Columnas finales: ['idparcela', 'coorx', 'coory', 'huso', 'presencia', 'tipsuelo1', 'tipsuelo2', 'tipsuelo3', 'rocosidad', 'radio', 'superficie']


comprobacion porque creo que he perdido info

In [18]:
import pandas as pd

# 1) Leer los CSVs necesarios
mapa4 = pd.read_csv("IFN_España/IFN4_campo/ifn4_campo_PCDatosMap.csv", usecols=['IDParcela','CoorX','CoorY','Huso','Presencia'])
mapa4.rename(columns={'IDParcela':'idparcela','CoorX':'coorx','CoorY':'coory','Huso':'huso','Presencia':'presencia'}, inplace=True)

mapa3 = pd.read_csv("IFN_España/IFN3_campo/ifn3_campo_PCDatosMap.csv", usecols=['IDParcela','CoorX','CoorY','Huso','Presencia'])
mapa3.rename(columns={'IDParcela':'idparcela','CoorX':'coorx','CoorY':'coory','Huso':'huso','Presencia':'presencia'}, inplace=True)

par4 = pd.read_csv("IFN_España/IFN4_campo/ifn4_campo_parcela.csv", usecols=['IDParcela','TipSuelo1','TipSuelo2','TipSuelo3','Rocosid','Radio','Superficie'])
par4.rename(columns={'IDParcela':'idparcela','TipSuelo1':'tipsuelo1','TipSuelo2':'tipsuelo2','TipSuelo3':'tipsuelo3','Rocosid':'rocosidad','Radio':'radio','Superficie':'superficie'}, inplace=True)


par3 = pd.read_csv("IFN_España/IFN3_campo/ifn3_campo_parcela.csv", usecols=['IDParcela','TipSuelo1','TipSuelo2','TipSuelo3','Rocosid','Radio','Superficie'])
par3.rename(columns={'IDParcela':'idparcela','TipSuelo1':'tipsuelo1','TipSuelo2':'tipsuelo2','TipSuelo3':'tipsuelo3','Rocosid':'rocosidad','Radio':'radio','Superficie':'superficie'}, inplace=True)

par_local = pd.read_csv("IFN_España/parcela.csv")

# 2) Unir los mapas: IFN4 prioritario, añadir IDs de IFN3 que no estén en IFN4
mapa_all = pd.concat([mapa4, mapa3[~mapa3['idparcela'].isin(mapa4['idparcela'])]], ignore_index=True)

# 3) Unir las columnas de par4 y par3 para completar los campos faltantes
par_tips = par4.set_index('idparcela').combine_first(par3.set_index('idparcela')).reset_index()

# 4) Combinar todo
parcela_enriquecida = mapa_all.set_index('idparcela').combine_first(par_tips.set_index('idparcela')).reset_index()

# 5) Contar filas nuevas respecto a par_local
id_nuevos = parcela_enriquecida[~parcela_enriquecida['idparcela'].isin(par_local['idparcela'])]
print(f"Filas totales: {len(parcela_enriquecida)}")
print(f"Filas nuevas añadidas: {len(id_nuevos)}")

# 6) Guardar resultado
parcela_enriquecida.to_csv("IFN_España/parcela_enriquecida.csv", index=False)


Filas totales: 94834
Filas nuevas añadidas: 53833


Finalmente eliminamos las que no estan en parcela

In [19]:
import pandas as pd
import numpy as np
import os

# --- Rutas ---
p_master = "IFN_España/parcela.csv"  # contiene las parcelas válidas
targets = [
    "IFN_España/parcela_inventario_especie_cd.csv",
    "IFN_España/parcela_inventario_especie.csv",
    "IFN_España/parcela_inventario.csv",
]

# --- Utilidad para normalizar idparcela ---
def clean_idparcela(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace("/.0$", "", regex=True)  # por si viene como "123.0"
    return s.replace({"": np.nan})

# --- Cargar parcelas válidas ---
keep_df = pd.read_csv(p_master, low_memory=False)
keep_df.columns = [c.strip().lower() for c in keep_df.columns]
if "idparcela" not in keep_df.columns:
    raise ValueError(f"[{p_master}] No se encontró la columna 'idparcela'.")

keep_ids = clean_idparcela(keep_df["idparcela"]).dropna().unique().tolist()
keep_ids_set = set(keep_ids)
print(f"Parcelas válidas en master ({p_master}): {len(keep_ids_set):,}")

# --- Procesar cada archivo destino ---
resumen = []

for path in targets:
    if not os.path.exists(path):
        print(f"[AVISO] No existe el archivo: {path}")
        continue

    df = pd.read_csv(path, low_memory=False)
    orig_cols = df.columns.tolist()
    df.columns = [c.strip().lower() for c in df.columns]

    if "idparcela" not in df.columns:
        raise ValueError(f"[{path}] No se encontró la columna 'idparcela'.")

    # Normaliza idparcela y filtra
    df["idparcela"] = clean_idparcela(df["idparcela"])
    n_before = len(df)
    n_nan_id = df["idparcela"].isna().sum()  # filas que no podrán casar
    mask_keep = df["idparcela"].isin(keep_ids_set)
    df_keep = df.loc[mask_keep].copy()
    n_after = len(df_keep)
    n_removed = n_before - n_after

    # Guardar (sobrescribe)
    df_keep.to_csv(path, index=False, encoding="utf-8-sig")

    print(f"\n[{os.path.basename(path)}]")
    print(f"  Filas originales: {n_before:,}")
    print(f"  Filas con idparcela NaN (se eliminan): {n_nan_id:,}")
    print(f"  Filas eliminadas por no estar en master: {n_removed:,}")
    print(f"  Filas restantes: {n_after:,}")

    resumen.append({
        "archivo": os.path.basename(path),
        "originales": n_before,
        "idparcela_nan": int(n_nan_id),
        "eliminadas": n_removed,
        "restantes": n_after
    })

# (Opcional) resumen en DataFrame
if resumen:
    resumen_df = pd.DataFrame(resumen)
    print("\n--- Resumen del cribado ---")
    print(resumen_df.to_string(index=False))


Parcelas válidas en master (IFN_España/parcela.csv): 41,001



[parcela_inventario_especie_cd.csv]
  Filas originales: 1,474,207
  Filas con idparcela NaN (se eliminan): 0
  Filas eliminadas por no estar en master: 566,575
  Filas restantes: 907,632



[parcela_inventario_especie.csv]
  Filas originales: 254,245
  Filas con idparcela NaN (se eliminan): 0
  Filas eliminadas por no estar en master: 91,835
  Filas restantes: 162,410



[parcela_inventario.csv]
  Filas originales: 139,889
  Filas con idparcela NaN (se eliminan): 0
  Filas eliminadas por no estar en master: 51,093
  Filas restantes: 88,796

--- Resumen del cribado ---
                          archivo  originales  idparcela_nan  eliminadas  restantes
parcela_inventario_especie_cd.csv     1474207              0      566575     907632
   parcela_inventario_especie.csv      254245              0       91835     162410
           parcela_inventario.csv      139889              0       51093      88796


pies_mayores-> parcela_arbol

In [20]:
# -*- coding: utf-8 -*-
from pathlib import Path
import pandas as pd

# Entradas / Salidas
IN_PATH  = Path("IFN_España/IFN4_sig/ifn4_sig_Mayores_exs.csv")   # tu IFN4 ya sin dups k=(IDParcela,OrdenIf3,Rumbo,Distanci)
OUT_PATH = Path("IFN_España/parcela_arbol4.csv")

# Mapeo solicitado (OrdenIf3 se elimina)
RENAME_MAP = {
    "Estrato": "estrato",
    "nArbol": "narbol",
    "Rumbo": "rumbo",
    "Distanci": "distancia",
    "CD": "cd",
    "Ht": "ht",
    "Dn1": "dn1",
    "Dn2": "dn2",
    "G": "g",
    "IAVC": "iavc",
    "VCC": "vcc",
    "VSC": "vsc",
    "VLE": "vle",
    "CA": "ca",
    "CR": "cr",
    "IDParcela": "parcela_id",
    "EspecieNormalizada": "especie_id",
}
FINAL_COL_ORDER = [
    "parcela_id","especie_id",
    "estrato","narbol","rumbo","distancia",
    "cd","ht","dn1","dn2","g","iavc","vcc","vsc","vle","ca","cr"
]

def leer_csv_robusto(ruta):
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, encoding=enc)
        except Exception:
            pass
    raise RuntimeError(f"No se pudo leer {ruta} con utf-8 ni latin-1")

def main():
    df = leer_csv_robusto(IN_PATH)

    # Asegurar columnas base
    cols_necesarias = set(RENAME_MAP.keys()) | {"OrdenIf3"}
    faltantes = [c for c in cols_necesarias if c not in df.columns]
    if faltantes:
        # OrdenIf3 puede no estar si ya venía eliminado en algún flujo; no es crítico.
        faltantes_sin_orden = [c for c in faltantes if c != "OrdenIf3"]
        if faltantes_sin_orden:
            raise ValueError(f"Faltan columnas requeridas en el IFN4 de entrada: {faltantes_sin_orden}")

    # Añadir CA/CR si faltaran para mantener esquema homogéneo
    for c in ("CA","CR"):
        if c not in df.columns:
            df[c] = pd.NA

    # Renombrar y eliminar OrdenIf3
    df = df.rename(columns=RENAME_MAP)
    if "OrdenIf3" in df.columns:
        df = df.drop(columns=["OrdenIf3"])

    # Reordenar columnas
    # (Si faltara alguna de las finales por venirse vacía, la creamos)
    for c in FINAL_COL_ORDER:
        if c not in df.columns:
            df[c] = pd.NA
    df = df[FINAL_COL_ORDER].copy()

    # Tipos básicos
    df["parcela_id"] = df["parcela_id"].astype(str)

    # Reasignar 'narbol' = enteros crecientes por parcela_id (en el orden actual)
    # Si prefieres un orden específico (p.ej. por estrato/rumbo/distancia), ordénalo antes del groupby.
    df["narbol"] = (
        df.groupby("parcela_id")
          .cumcount() + 1
    ).astype("Int64")

    # Validación: unicidad (parcela_id, narbol)
    clave_rep = (
        df.groupby(["parcela_id","narbol"], as_index=False)
          .size()
          .query("size > 1")
    )
    if not clave_rep.empty:
        top = clave_rep.head(10).to_dict("records")
        raise AssertionError(
            f"(parcela_id, narbol) no es único tras la reasignación. Ejemplos: {top}"
        )

    # Guardar y mostrar resumen
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT_PATH, index=False)

    print("== Transformación IFN4 completada ==")
    print(f"Entrada : {IN_PATH}")
    print(f"Salida  : {OUT_PATH}")
    print(f"Filas   : {len(df):,}")
    # Vista rápida
    print("\nMuestra (5 filas):")
    print(df.head(5).to_string(index=False))

    # Estadística corta por parcela (distribución del número de árboles)
    conteo = df.groupby("parcela_id")["narbol"].max().describe()
    print("\nÁrboles por parcela (descriptivos sobre 'narbol' max por parcela):")
    print(conteo.round(2))

if __name__ == "__main__":
    pd.set_option("display.max_colwidth", 200)
    main()


== Transformación IFN4 completada ==
Entrada : IFN_España/IFN4_sig/ifn4_sig_Mayores_exs.csv
Salida  : IFN_España/parcela_arbol4.csv
Filas   : 863,127

Muestra (5 filas):
parcela_id  especie_id  estrato  narbol  rumbo  distancia  cd  ht  dn1  dn2        g     iavc        vcc        vsc       vle        ca        cr
     19_03          21        9       1      2        9.6  15 8.3  153  155 0.018626 5.049952  76.143900  55.479214  5.921179 28.564860 15.415400
     19_03          21        9       2      5        8.8  15 8.0  166  171 0.022299 5.465844  87.992857  65.236541  7.182733 34.743362 18.454963
     19_03          21        9       3      6        7.8  15 7.0  141  141 0.015614 4.682594  44.463839  34.240229  4.899996 23.810327 12.922650
     19_03          21        9       4     13        7.2  20 7.7  181  185 0.026302 5.888803  85.597413  65.920228  8.575165 41.710959 21.767850
     19_03          21        9       5     23        6.2  20 8.7  226  212 0.037668 6.973397 158.83

estudio faltantes del archivo y creo un narbol_id robusto

In [21]:
# -*- coding: utf-8 -*-
# Estudia y normaliza 'IFN_España/parcela_arbol4.csv':
# - Renombra narbol -> narbol_id
# - Elimina filas con parcela_id o especie_id faltantes (o vacíos)
# - Reasigna narbol_id = 1..N por (parcela_id, especie_id) en orden determinista
# - Valida unicidad (parcela_id, especie_id, narbol_id)
# - Imprime conteo de faltantes por columna

from pathlib import Path
import pandas as pd
import numpy as np

PATH = Path("IFN_España/parcela_arbol4.csv")  # entrada
# SAVE_PATH = Path("IFN_España/parcela_arbol4_limpio.csv")  # descomentar para guardar

COLS = ["parcela_id","especie_id","estrato","narbol","rumbo","distancia","cd","ht","dn1","dn2","g","iavc","vcc","vsc","vle","ca","cr"]

def leer_csv_robusto(p: Path) -> pd.DataFrame:
    for enc in ("utf-8-sig","latin-1"):
        try:
            return pd.read_csv(p, encoding=enc)
        except Exception:
            pass
    raise RuntimeError(f"No pude leer {p} con utf-8 ni latin-1.")

def main():
    df = leer_csv_robusto(PATH)

    # --- Comprobaciones/normalización mínima ---
    falt = [c for c in COLS if c not in df.columns]
    if falt:
        raise ValueError(f"Faltan columnas en {PATH}: {falt}")

    df = df[COLS].copy()

    # Normalizar ids
    df["parcela_id"] = df["parcela_id"].astype(str).str.strip()
    df["especie_id"] = df["especie_id"].astype(str).str.strip().str.lower()

    # Considerar vacíos como NaN para filtrar
    df["parcela_id"].replace({"": pd.NA}, inplace=True)
    df["especie_id"].replace({"": pd.NA}, inplace=True)

    # Tipos numéricos
    num_cols = ["estrato","narbol","rumbo","distancia","cd","ht","dn1","dn2","g","iavc","vcc","vsc","vle","ca","cr"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["estrato"] = df["estrato"].astype("Int64")
    df["narbol"]  = df["narbol"].astype("Int64")

    # --- Eliminar filas con parcela_id/especie_id faltantes ---
    n_before = len(df)
    df = df.dropna(subset=["parcela_id","especie_id"]).copy()
    dropped = n_before - len(df)

    # --- Renombrar narbol -> narbol_id ---
    df = df.rename(columns={"narbol": "narbol_id"})

    # --- Reasignar narbol_id = 1..N por (parcela_id, especie_id) ---
    # Orden determinista para la numeración
    df = df.sort_values(
        ["parcela_id","especie_id","estrato","rumbo","distancia","narbol_id"],
        kind="mergesort"
    ).reset_index(drop=True)

    df["narbol_id"] = (df.groupby(["parcela_id","especie_id"]).cumcount() + 1).astype("Int64")

    # --- Validaciones ---
    # 1) Unicidad de (parcela_id, especie_id, narbol_id)
    dup_rows = df.duplicated(subset=["parcela_id","especie_id","narbol_id"], keep=False)
    n_dup = int(dup_rows.sum())

    # 2) Chequeo 1..N exacto por grupo
    grp = df.groupby(["parcela_id","especie_id"])["narbol_id"]
    bad_groups = (grp.min() != 1) | (grp.max() != grp.nunique())
    n_bad = int(bad_groups.sum())

    # --- Conteo de faltantes por columna ---
    miss = df.isna().sum().sort_values(ascending=False)

    # --- Salidas por consola ---
    print("== Normalización parcela_arbol4 ==")
    print(f"Archivo: {PATH}")
    print(f"Filas originales         : {n_before:,}")
    print(f"Filas sin parcela/especie: {dropped:,} (eliminadas)")
    print(f"Filas finales            : {len(df):,}")

    print("\n== Validaciones ==")
    print(f"Duplicados en (parcela_id, especie_id, narbol_id): {n_dup:,}")
    if n_dup:
        print("Ejemplo de claves duplicadas (top 10):")
        print(df.loc[dup_rows, ["parcela_id","especie_id","narbol_id"]]
                .value_counts().reset_index(name="reps").head(10).to_string(index=False))
    print(f"Grupos que no son 1..N exacto: {n_bad:,}")

    print("\n== Conteo de faltantes por columna ==")
    print(miss.to_string())

    print("\nMuestra (5 filas):")
    print(df.head(5).to_string(index=False))

    # --- Guardar (opcional) ---
    # SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    # df.to_csv(SAVE_PATH, index=False)
    # print(f"\nArchivo guardado en: {SAVE_PATH}")

if __name__ == "__main__":
    pd.set_option("display.max_colwidth", 200)
    main()


/tmp/ipykernel_2046064/1130913459.py:41: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["parcela_id"].replace({"": pd.NA}, inplace=True)
/tmp/ipykernel_2046064/1130913459.py:42: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method nev

== Normalización parcela_arbol4 ==
Archivo: IFN_España/parcela_arbol4.csv
Filas originales         : 863,127
Filas sin parcela/especie: 0 (eliminadas)
Filas finales            : 863,127

== Validaciones ==
Duplicados en (parcela_id, especie_id, narbol_id): 0
Grupos que no son 1..N exacto: 0

== Conteo de faltantes por columna ==
cr            381123
ca            381123
parcela_id         0
especie_id         0
estrato            0
distancia          0
cd                 0
narbol_id          0
rumbo              0
dn1                0
ht                 0
dn2                0
g                  0
vcc                0
iavc               0
vle                0
vsc                0

Muestra (5 filas):
parcela_id especie_id  estrato  narbol_id  rumbo  distancia  cd   ht  dn1  dn2        g      iavc        vcc        vsc       vle  ca  cr
     01_03        215       35          1    227        9.2  15  5.0  151  168 0.019981  1.966990  40.911456  29.794673 13.395243 NaN NaN
     01_03      

In [22]:
# -*- coding: utf-8 -*-
"""
Filtra 'IFN_España/parcela_arbol4.csv' para conservar solo las filas
cuyo 'especie_id' está presente en 'especies_to_load.csv' (columna 'especie').
Si el archivo de especies no existe o está vacío, conserva todas las filas.
"""

from pathlib import Path
import pandas as pd
import os

# Rutas de entrada/salida
IN_FILE  = Path("IFN_España/parcela_arbol4.csv")
ESP_FILE = Path("especies_to_load.csv")
OUT_FILE = Path("IFN_España/parcela_especie_arbol.csv")

def main():
    df = pd.read_csv(IN_FILE, low_memory=False)
    total_ini = len(df)
    
    # Usar especies de referencia si el archivo existe y tiene datos
    if ESP_FILE.exists():
        especies_ref = pd.read_csv(ESP_FILE)
        if len(especies_ref) > 0 and "especie" in especies_ref.columns:
            df["especie_id"] = df["especie_id"].astype(str).str.strip().str.lower()
            especies_ref["especie"] = especies_ref["especie"].astype(str).str.strip().str.lower()
            especies_validas = set(especies_ref["especie"].dropna().unique())
            
            mask_valid = df["especie_id"].isin(especies_validas)
            df_filtrado = df[mask_valid].copy()
            print(f"== Filtrado por {len(especies_validas):,} especies válidas ==")
        else:
            print("⚠️  especies_to_load.csv está vacío → conservando todas las filas")
            df_filtrado = df.copy()
    else:
        print("⚠️  especies_to_load.csv no encontrado → conservando todas las filas")
        df_filtrado = df.copy()
    
    total_fin = len(df_filtrado)
    print(f"Filas originales: {total_ini:,}")
    print(f"Filas restantes: {total_fin:,}")
    print(f"Filas eliminadas: {total_ini - total_fin:,}")

    # Guardar
    OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    df_filtrado.to_csv(OUT_FILE, index=False)
    print(f"\nArchivo guardado en: {OUT_FILE}")

pd.set_option("display.max_colwidth", 200)
main()


⚠️  especies_to_load.csv está vacío → conservando todas las filas
Filas originales: 863,127
Filas restantes: 863,127
Filas eliminadas: 0



Archivo guardado en: IFN_España/parcela_especie_arbol.csv


Edicion de Especies.csv


In [23]:
# -*- coding: utf-8 -*-
import pandas as pd
import os
from pathlib import Path

INPUT_XLSX = "especies.xlsx"
OUTPUT_CSV = "especies_to_load.csv"

if not os.path.exists(INPUT_XLSX):
    print(f"⚠️  {INPUT_XLSX} no encontrado. Saltando celda de especies.")
    # Crear CSV vacío con las columnas esperadas
    pd.DataFrame(columns=["especie", "nombre", "sinonimia", "conif_fronf"]).to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"CSV vacío creado en: {OUTPUT_CSV}")
else:
    def safe_int(x):
        if pd.isna(x):
            return None
        try:
            return int(float(x))
        except Exception:
            import re
            m = re.search(r"-?\d+", str(x))
            return int(m.group(0)) if m else None

    def normalize_text(x):
        if pd.isna(x):
            return None
        s = str(x).strip()
        return s if s != "" else None

    def concat_unique(series):
        vals = [str(v).strip() for v in series.dropna().astype(str)]
        import re
        exploded = []
        for s in vals:
            parts = re.split(r"[;,|/]+", s)
            for p in parts:
                if p.strip():
                    exploded.append(p.strip())
        seen, out = set(), []
        for x in exploded:
            if x not in seen:
                seen.add(x)
                out.append(x)
        return " | ".join(out) if out else None

    print("📖 Leyendo Excel:", INPUT_XLSX)
    df = pd.read_excel(INPUT_XLSX, engine="openpyxl", dtype=object)

    expected_map = {
        'clave': 'especie',
        'nombre_especie': 'nombre',
        'sinonimias': 'sinonimia',
        'conif_fronf': 'conif_fronf'
    }
    missing = [c for c in expected_map if c not in df.columns]
    if missing:
        raise SystemExit(f"❌ Faltan columnas en el Excel: {missing}")

    df_proc = pd.DataFrame()
    df_proc['especie'] = df['clave'].apply(safe_int)
    df_proc['nombre'] = df['nombre_especie'].apply(normalize_text)
    df_proc['sinonimia'] = df['sinonimias'].apply(normalize_text)
    df_proc['conif_fronf'] = df['conif_fronf']

    df_proc = df_proc[df_proc['especie'].notnull()].copy()

    grouped = df_proc.groupby('especie', sort=False).agg({
        'nombre': lambda s: next((x for x in s if pd.notna(x)), None),
        'sinonimia': concat_unique,
        'conif_fronf': lambda s: next((x for x in s if pd.notna(x)), None)
    }).reset_index()

    grouped.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
    print("✅ CSV creado con conif_fronf:", Path(OUTPUT_CSV).resolve())


⚠️  especies.xlsx no encontrado. Saltando celda de especies.
CSV vacío creado en: especies_to_load.csv


In [24]:
# -*- coding: utf-8 -*-
import pandas as pd

# --- Carga del archivo ---
df = pd.read_csv("especies_to_load.csv", dtype=str)

# --- Diccionarios de grupos y claves (usa los que pasaste) ---
grupos_diccionario = { 
    "Pinos": ["Pinus sylvestris", "Pinus uncinata", "Pinus pinea", "Pinus halepensis",
              "Pinus nigra", "Pinus pinaster", "Pinus canariensis", "Pinus radiata", "Otros pinos"],
    "Abies": ["Abies alba", "Abies pinsapo"],
    "Larix": ["Larix spp.", "Larix decidua", "Larix leptolepis", "Larix x eurolepis"],
    "Cupressus": ["Cupressus sempervirens", "Cupressus arizonica", "Cupressus lusitanica",
                  "Cupressus macrocarpa", "Cupressus spp."],
    "Juniperus": ["Juniperus communis", "Juniperus thurifera", "Juniperus phoenicea",
                  "Juniperus oxycedrus", "Juniperus turbinata", "Juniperus sabina",
                  "Juniperus cedrus", "Juniperus spp."],
    "Quercus": ["Quercus robur", "Quercus petraea", "Quercus pyrenaica", "Quercus faginea",
                "Quercus ilex ssp. ballota", "Quercus ilex ssp. ilex", "Quercus suber",
                "Quercus canariensis", "Quercus rubra", "Quercus pubescens", "Quercus lusitanica",
                "Quercus alpestris", "Otros quercus"],
    "Salix": ["Salix spp.", "Salix alba", "Salix atrocinerea", "Salix x canadensis",
              "Salix cantabrica", "Salix elaeagnos", "Salix fragilis", "Salix canariensis",
              "Salix purpurea", "Salix caprea", 'Salix alba'],
    "Populus": ["Populus alba", "Populus tremula", "Populus nigra", "Populus x canadensis"],
    "Fraxinus": ["Fraxinus angustifolia", "Fraxinus excelsior", "Fraxinus ornus", "Fraxinus spp."],
    "Eucalyptus": ["Eucalyptus globulus", "Eucalyptus camaldulensis", "Eucalyptus nitens",
                   "Eucalyptus viminalis", "Eucalyptus gomphocephalus", "Eucalyptus robusta",
                   "Otros eucaliptos"],
    "Acer": ["Acer campestre", "Acer monspessulanum", "Acer negundo", "Acer opalus",
             "Acer platanoides", "Acer pseudoplatanus", "Acer spp."],
    "Tilia": ["Tilia spp.", "Tilia cordata", "Tilia platyphyllos"],
    "Sorbus": ["Sorbus spp.", "Sorbus aria", "Sorbus aucuparia", "Sorbus domestica",
               "Sorbus torminalis", "Sorbus latifolia", "Sorbus chamaemespilus"],
    "Morus": ["Morus spp.", "Morus alba", "Morus nigra"],
    "Sambucus": ["Sambucus nigra", "Sambucus racemosa", "Sambucus spp."],
    "Juglans": ["Juglans regia", "Juglans nigra", "Juglans spp."],
    "Crataegus": ["Crataegus spp.", "Crataegus monogyna", "Crataegus laevigata",
                  "Crataegus laciniata", "Crataegus azarolus"],
    "Prunus": ["Prunus spp.", "Prunus spinosa", "Prunus avium", "Prunus lusitanica", "Prunus padus"],
    "Tamarix": ["Tamarix spp.", "Tamarix canariensis"],
    "Cedrus": ["Cedrus spp.", "Juniperus cedrus", 'Cedrus libani', 'Cedrus deodara', 'Cedrus atlantica'],
    "Acacia": ["Acacia spp.", "Robinia pseudoacacia", 'Acacia melanoxylon', 'Sophora japonica', 'Acacia dealbata', 'Gleditsia triacanthos'],
    'Ulmus': ['Ulmus spp.','Ulmus minor', 'Ulmus glabra','Ulmus pumila'],
    'Betula': ['Betula spp.', 'Betula alba', 'Betula pendula'],
    'Coníferas': ['Otras coníferas','Mezcla de coníferas'],
    'Ilex': ['Ilex aquifolium','Ilex canariensis','Ilex platyphylla'],
    'Frondosas': ['Otras frondosas','Mezcla de frondosas de gran porte', 'Mezcla de pequeñas frondosas'],
    "Arbutus": ["Arbutus unedo", "Arbutus canariensis"],
    "Phoenix": ["Phoenix spp.", "Phoenix canariensis"],
    "Platanus": ["Platanus hispanica", "Platanus orientalis"],
    "Laurisilva": ["Laurisilva", "Otras laurisilvas"],
    "Buxus": ["Buxus sempervirens", "Buxus balearica"],
    "Pistacia": ["Pistacia terebinthus", "Pistacia atlantica"],
    "Laurus": ["Laurus nobilis", "Laurus azorica"]
}

grupo_to_clave = {
    'Frondosas':99, 'Ilex':65, 'Salix':57, 'Coníferas':19, 'Betula':73,'Ulmus':956,
    "Acacia":7, "Pinos": 20, "Abies": 31, "Larix": 35, 
    "Cupressus": 936, "Juniperus": 937, "Quercus": 40, "Populus": 58, 
    "Fraxinus": 455, "Eucalyptus": 60, "Acer": 976, "Tilia": 77, 
    "Sorbus": 78, "Morus": 399, "Sambucus": 997, "Juglans": 975, 
    "Crataegus": 15, "Prunus": 95, "Tamarix": 53, "Cedrus": 917,
    "Arbutus": 68, "Phoenix": 69, "Platanus": 79, "Laurisilva": 80,
    "Buxus": 91, "Pistacia": 93, "Laurus": 94
}

# --- Crear mapeo especie -> grupo ---
especie_to_grupo = {}
for grupo, lista in grupos_diccionario.items():
    for especie in lista:
        especie_to_grupo[especie] = grupo

# --- Función de asignación ---
def asignar_grupo(nombre):
    grupo = especie_to_grupo.get(nombre)
    if grupo:
        return pd.Series([grupo, grupo_to_clave.get(grupo)])
    else:
        return pd.Series([pd.NA, pd.NA])

if len(df) == 0:
    df['nombre_grupo'] = pd.Series(dtype='object')
    df['especie_grupo'] = pd.Series(dtype='object')
    print("⚠️  especies_to_load.csv está vacío, saltando asignación de grupos.")
else:
    df[['nombre_grupo','especie_grupo']] = df['nombre'].apply(asignar_grupo)

# --- Guardar resultado ---
df.to_csv("especies_to_load.csv", index=False, encoding="utf-8")

print("Archivo guardado con las columnas 'nombre_grupo' y 'especie_grupo'")
print(df.head(15))


⚠️  especies_to_load.csv está vacío, saltando asignación de grupos.
Archivo guardado con las columnas 'nombre_grupo' y 'especie_grupo'
Empty DataFrame
Columns: [especie, nombre, sinonimia, conif_fronf, nombre_grupo, especie_grupo]
Index: []
